<a href="https://colab.research.google.com/github/ArrushiTripathi2429/CutMix-Implementation/blob/main/CutMix_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [4]:
from torchvision import datasets

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True
)

print("Original training images:", len(train_dataset))
print("Original testing images:", len(test_dataset))

100%|██████████| 170M/170M [00:13<00:00, 12.2MB/s]


Original training images: 50000
Original testing images: 10000


In [ ]:
print("Classes:")
print(train_dataset.classes)

print("\nNumber of classes:")
print(len(train_dataset.classes))

# Check one image
image, label = train_dataset[0]

print("\nImage shape:", image.size)
print("Label number:", label)
print("Label name:", train_dataset.classes[label])

Classes:
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Number of classes:
10

Image shape: (32, 32)
Label number: 6
Label name: frog


In [5]:
from torch.utils.data import Subset
import numpy as np

# Number of images per class
TRAIN_PER_CLASS = 300
TEST_PER_CLASS = 200

# Get labels
train_targets = np.array(train_dataset.targets)
test_targets = np.array(test_dataset.targets)

# Store selected indices
train_indices = []
test_indices = []

# Select images class by class
for class_id in range(10):

    # Indices belonging to this class
    train_class_indices = np.where(train_targets == class_id)[0]
    test_class_indices = np.where(test_targets == class_id)[0]

    # Shuffle indices
    np.random.shuffle(train_class_indices)
    np.random.shuffle(test_class_indices)

    # Take required number
    train_indices.extend(train_class_indices[:TRAIN_PER_CLASS])
    test_indices.extend(test_class_indices[:TEST_PER_CLASS])

# Create subsets
train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

print("Training subset:", len(train_subset))
print("Testing subset:", len(test_subset))

Training subset: 3000
Testing subset: 2000


In [6]:
from collections import Counter

train_selected_labels = [
    train_dataset.targets[i] for i in train_indices
]

test_selected_labels = [
    test_dataset.targets[i] for i in test_indices
]

print("Training distribution:")
for class_id, count in sorted(Counter(train_selected_labels).items()):
    print(train_dataset.classes[class_id], ":", count)

print("\nTesting distribution:")
for class_id, count in sorted(Counter(test_selected_labels).items()):
    print(test_dataset.classes[class_id], ":", count)

Training distribution:
airplane : 300
automobile : 300
bird : 300
cat : 300
deer : 300
dog : 300
frog : 300
horse : 300
ship : 300
truck : 300

Testing distribution:
airplane : 200
automobile : 200
bird : 200
cat : 200
deer : 200
dog : 200
frog : 200
horse : 200
ship : 200
truck : 200


In [11]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform
)

In [12]:
from torch.utils.data import Subset

train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

print("Training subset:", len(train_subset))
print("Testing subset:", len(test_subset))

Training subset: 3000
Testing subset: 2000


In [13]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [14]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([64, 3, 32, 32])
Labels shape: torch.Size([64])
